In [7]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split




In [2]:
df = pd.read_csv('heart.csv')

print(df.sample(6))

     age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  \
142   42    0   2       120   209    0        1      173      0      0.0   
49    53    0   0       138   234    0        0      160      0      0.0   
28    65    0   2       140   417    1        0      157      0      0.8   
253   67    1   0       100   299    0        0      125      1      0.9   
108   50    0   1       120   244    0        1      162      0      1.1   
175   40    1   0       110   167    0        0      114      1      2.0   

     slope  ca  thal  target  
142      1   0     2       1  
49       2   0     2       1  
28       2   1     2       1  
253      1   2     2       0  
108      2   0     2       1  
175      1   0     3       0  


In [4]:
missing_values = df.isnull().sum()
print("Missing values in each column:\n", missing_values)

numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
categorical_cols = df.select_dtypes(include=['object']).columns

for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].mean(), inplace=True)

for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].mode()[0], inplace=True)

print("\nRemaining missing values (should be zero):\n", df.isnull().sum().sum())

Missing values in each column:
 age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
target      0
dtype: int64

Remaining missing values (should be zero):
 0


In [ ]:
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
categorical_cols = df.select_dtypes(include=['object']).columns

for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].mean(), inplace=True)

for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].mode()[0], inplace=True)


In [6]:
scaler = StandardScaler()
scaled_features = scaler.fit_transform(df.drop('target', axis=1))

import numpy as np
X = pd.DataFrame(scaled_features, columns=df.drop('target', axis=1).columns)
y = df['target']

In [8]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score

model_lr = LogisticRegression()
model_rf = RandomForestClassifier(random_state=42)
model_svc = SVC(probability=True)

model_lr.fit(X_train, y_train)
model_rf.fit(X_train, y_train)
model_svc.fit(X_train, y_train)

pred_lr = model_lr.predict(X_test)
pred_rf = model_rf.predict(X_test)
pred_svc = model_svc.predict(X_test)

acc_lr = accuracy_score(y_test, pred_lr)
acc_rf = accuracy_score(y_test, pred_rf)
acc_svc = accuracy_score(y_test, pred_svc)

f1_lr = f1_score(y_test, pred_lr)
f1_rf = f1_score(y_test, pred_rf)
f1_svc = f1_score(y_test, pred_svc)

results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'SVM'],
    'Accuracy': [acc_lr, acc_rf, acc_svc],
    'F1 Score': [f1_lr, f1_rf, f1_svc]
})

print(results)


                 Model  Accuracy  F1 Score
0  Logistic Regression  0.803279  0.833333
1        Random Forest  0.836066  0.864865
2                  SVM  0.836066  0.861111


In [20]:
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score

base_learners = [
    ('xgb', XGBClassifier(eval_metric='logloss', random_state=42)),
    ('knn', KNeighborsClassifier()),
    ('lr', LogisticRegression())
]

meta_model = SVC(random_state=42)

stack_model = StackingClassifier(estimators=base_learners, final_estimator=meta_model, cv=5)
stack_model.fit(X_train, y_train)
y_pred_stack = stack_model.predict(X_test)

acc_stack = accuracy_score(y_test, y_pred_stack)
f1_stack = f1_score(y_test, y_pred_stack)

results.loc[results['Model'] == 'Stacked Model', ['Accuracy', 'F1 Score']] = [acc_stack, f1_stack]

print("Final Model Comparison (Random Forest as meta-model):\n")
print(results.to_string(index=False))


Final Model Comparison (Random Forest as meta-model):

              Model  Accuracy  F1 Score
Logistic Regression  0.803279  0.833333
      Random Forest  0.836066  0.864865
                SVM  0.836066  0.861111
      Stacked Model  0.819672  0.853333
